# 04 - Model

Khởi tạo model theo `configs/model.yaml` (đã chốt: **YOLOv8s**, pretrained COCO, input 1280x960 - xem `docs/architecture.md`).
Kiểm tra kiến trúc, xác nhận GPU hoạt động, chạy smoke test forward pass trên ảnh thật từ `data/processed/`,
và đối chiếu cấu hình model với `data.yaml` sinh ra ở `03_preprocessing.ipynb` trước khi sang bước training.

## 0. Setup

Đọc `configs/model.yaml`, thêm project root vào `sys.path` để import `src/models/`.

In [1]:
import os
import sys

import yaml
import torch

sys.path.insert(0, os.path.abspath(".."))
from src.models.model_factory import build_model

with open("../configs/model.yaml", encoding="utf-8") as f:
    model_cfg = yaml.safe_load(f)

print("Model config:", model_cfg)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Model config: {'name': 'yolov8', 'variant': 's', 'task': 'detection', 'num_classes': 1, 'class_names': ['Human'], 'pretrained': True, 'input_channels': 3, 'input_size': [640, 640]}
CUDA available: True
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


## 1. Xây dựng model

Load checkpoint YOLOv8s pretrained trên COCO qua `src/models/model_factory.py` (factory cho phép đổi kiến trúc sau này chỉ bằng cách sửa `configs/model.yaml`, không sửa code).

In [2]:
weights_dir = os.path.abspath("../outputs/checkpoints")
model = build_model(model_cfg, weights_dir=weights_dir)
print("Loại model:", type(model))
print("Task:", model.task)
print("Checkpoint cache tại:", weights_dir)
model.info()

Loại model: <class 'ultralytics.models.yolo.model.YOLO'>
Task: detect
Checkpoint cache tại: D:\Thermal_project\outputs\checkpoints


YOLOv8s summary: 129 layers, 11,166,560 parameters, 0 gradients, 28.8 GFLOPs


(129, 11166560, 0, 28.816844800000002)

## 2. Smoke test: forward pass trên ảnh thật

Chạy thử `predict` trên 1 ảnh từ `data/processed/images/train` (đã qua denoise+CLAHE ở notebook 03) để xác nhận model chạy được trên GPU, không lỗi kích thước/kênh ảnh.

**Lưu ý quan trọng**: checkpoint hiện tại là pretrained COCO (80 class, gồm class `person`) - **chưa fine-tune** theo dataset của dự án, nên kết quả predict ở đây chỉ để kiểm tra pipeline chạy được, không phản ánh độ chính xác thật. Việc fine-tune về 1 class `Human` sẽ làm ở `05_training.ipynb`.

In [3]:
import random

train_dir = os.path.abspath("../data/processed/images/train")
sample_images = os.listdir(train_dir)
if not sample_images:
    raise RuntimeError("Chưa có ảnh trong data/processed/images/train. Chạy notebook 03_preprocessing.ipynb trước.")

sample_path = os.path.join(train_dir, random.choice(sample_images))
device = 0 if torch.cuda.is_available() else "cpu"

results = model.predict(source=sample_path, device=device, imgsz=model_cfg["input_size"][0], verbose=False)
r = results[0]
print("Ảnh test:", sample_path)
print("Kích thước ảnh gốc:", r.orig_shape)
print("Số box phát hiện (checkpoint COCO, chưa fine-tune):", len(r.boxes))
print("Device model đang chạy:", next(model.model.parameters()).device)

Ảnh test: D:\Thermal_project\data\processed\images\train\3079.jpg
Kích thước ảnh gốc: (960, 1280)
Số box phát hiện (checkpoint COCO, chưa fine-tune): 1
Device model đang chạy: cuda:0


## 3. Đối chiếu cấu hình model với `data.yaml`

Xác nhận `num_classes`/`class_names` trong `configs/model.yaml` khớp với `nc`/`names` trong `data/processed/data.yaml` (sinh ra ở notebook 03) - tránh lỗi lệch số class khi train.

In [4]:
data_yaml_path = os.path.abspath("../data/processed/data.yaml")
with open(data_yaml_path, encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

print("data.yaml:", data_cfg)

assert data_cfg["nc"] == model_cfg["num_classes"], (
    f"Lệch số class: data.yaml nc={data_cfg['nc']} != model.yaml num_classes={model_cfg['num_classes']}"
)
assert data_cfg["names"] == model_cfg["class_names"], (
    f"Lệch tên class: data.yaml names={data_cfg['names']} != model.yaml class_names={model_cfg['class_names']}"
)
print("Khớp: num_classes/class_names giữa configs/model.yaml và data/processed/data.yaml.")

data.yaml: {'path': 'D:/Thermal_project/data/processed', 'train': 'images/train', 'val': 'images/val', 'test': 'images/test', 'nc': 1, 'names': ['Human']}
Khớp: num_classes/class_names giữa configs/model.yaml và data/processed/data.yaml.


## 4. Tóm tắt

In [5]:
print("=" * 60)
print("TÓM TẮT MODEL")
print("=" * 60)
print(f"Kiến trúc: YOLOv8{model_cfg['variant']} (pretrained={model_cfg['pretrained']})")
print(f"Input size: {model_cfg['input_size']}")
print(f"Số class (fine-tune sắp tới): {model_cfg['num_classes']} ({model_cfg['class_names']})")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'không có, chạy CPU'}")
print("Model đã load thành công, forward pass smoke test OK, config khớp data.yaml.")
print("=> Sẵn sàng cho 05_training.ipynb (fine-tune trên data/processed/data.yaml).")

TÓM TẮT MODEL
Kiến trúc: YOLOv8s (pretrained=True)
Input size: [640, 640]
Số class (fine-tune sắp tới): 1 (['Human'])
GPU: NVIDIA GeForce RTX 3060 Laptop GPU
Model đã load thành công, forward pass smoke test OK, config khớp data.yaml.
=> Sẵn sàng cho 05_training.ipynb (fine-tune trên data/processed/data.yaml).
